<a href="https://colab.research.google.com/github/MITS-CAI/DL/blob/main/12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
# Upload a Google stock CSV file in Colab
from google.colab import files
uploaded = files.upload()
# Read the uploaded CSV file
df = pd.read_csv(list(uploaded.keys())[0])
# Select closing prices
data = df[["Close"]].dropna().values
# Normalize prices
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)
# Create sequences using the previous 60 trading days
x, y = [], []
for i in range(60, len(scaled_data)):
 x.append(scaled_data[i-60:i, 0])
 y.append(scaled_data[i, 0])
x = np.array(x)
y = np.array(y)
# Reshape input for LSTM: samples, time steps, features
x = x.reshape(x.shape[0], x.shape[1], 1)
# Split data in time order (do not shuffle)
split = int(0.8 * len(x))
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]
# Create LSTM model
model = Sequential([
 Input(shape=(60, 1)),
 LSTM(50, return_sequences=True),
 LSTM(50),
 Dense(1)
])
# Compile model
model.compile(optimizer="adam", loss="mean_squared_error")
# Train model
model.fit(x_train, y_train, epochs=5, batch_size=32)
# Predict test prices
predictions = model.predict(x_test)
# Convert predictions and actual values to original prices
predictions = scaler.inverse_transform(predictions)
actual = scaler.inverse_transform(y_test.reshape(-1, 1))
# Predict the next available trading day's closing price
last_60_days = scaled_data[-60:].reshape(1, 60, 1)
next_scaled_price = model.predict(last_60_days)
next_price = scaler.inverse_transform(next_scaled_price)
last_price = data[-1][0]
print("Last Available Closing Price:", last_price)
print("Predicted Next Trading Day Price:", next_price[0][0])
if next_price[0][0] > last_price:
 print("Trend: Increasing")
elif next_price[0][0] < last_price:
 print("Trend: Decreasing")
else:
 print("Trend: No Change")
# Plot actual and predicted prices
plt.plot(actual, label="Actual Price")
plt.plot(predictions, label="Predicted Price")
plt.xlabel("Test Time Steps")
plt.ylabel("Stock Price")
plt.title("Google Stock Price Prediction")
plt.legend()
plt.show()


KeyboardInterrupt: 